# Retrieval Quality — Why RAG Still Hallucinates
## Problem Statement
Build a retrieval audit pipeline: test whether the ChromaDB retriever finds the right rule for 5 known bugs, compute Precision@3, and fix one failure case.

## My Approach
* My plan was to treat the retrieval audit like a unit test suite — each bug query has an expected "pass condition" (the ground truth keyword), and I'd check if ChromaDB's top-3 results satisfied it.
* For embedding, I used the HuggingFace Inference API with all-mpnet-base-v2 (768-dim) since my existing collection was built with that model. I pre-computed all 5 query embeddings in one batch call so I could reuse them for both the before and after audit passes.
* If a ground truth keyword was missing from all top-3 results, my hypothesis was that the rule simply didn't exist in the collection yet — not that the query was wrong. So my fix strategy was: add targeted rules first, re-run, and only rephrase the query if it still missed.

## What I Learned
* The biggest insight was that distance > 1.0 in ChromaDB is a red flag — it means the retriever found nothing genuinely similar and is returning the least-bad option. That's not retrieval working, that's retrieval failing silently.
* I also learned that keyword-based hit detection has limits. For Bug 5, the retrieved chunk was semantically correct (it was about swallowing exceptions), but my ground truth keyword "exception handling" didn't appear in the rule text. The retriever was right; my evaluation was wrong. That's a real lesson about how RAG evaluation can lie to you even when the system is working.
* Finally — ID management in ChromaDB matters more than you'd think. When I tried adding new rules naively, they silently failed because the IDs already existed. Getting the last ID first and incrementing from there is the right pattern for append-only collections.

## Where I Got Stuck
* New rules not getting added. My first attempt used rule_0, rule_1 etc. which collided with existing IDs. ChromaDB doesn't raise an error on duplicate IDs — it just ignores the add. I only caught this by printing the collection count before and after. Fix: fetch the last ID, extract the number, and start from last_num + 1.
* Case-sensitive ground truth matching. One bug was showing a MISS even though the correct rule was clearly in the results. The rule text used "Mutable" (capital M) but my ground truth keyword was "mutable default" (lowercase). The fix was one line — .lower() on the document before checking — but it took me a few minutes to spot. 

## What I'd Do Differently
Longer term: for distance thresholds, I want to add logic that returns "No relevant rule found" when the best match distance is above 1.0 — rather than letting the LLM hallucinate an answer from whatever chunk is least-bad.

In [ ]:
# Setup + Bug Definitions
import chromadb
from dotenv import load_dotenv
from huggingface_hub import login
import os
import requests
import numpy as np
load_dotenv()

HF_TOKEN = os.getenv("HUGGING_FACE_API_KEY")

model_ids = {
    "MPNet": "all-mpnet-base-v2"
}


In [8]:
# Loading embedding-models collection
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection("api_driven_rules")

# 5 bug scenarios and their retrieval queries
BUG_QUERIES = {
    "bug_1_mutable_default": "mutable default argument Python function",
    "bug_2_bare_except":     "bare except clause Python anti-pattern",
    "bug_3_nested_loop":     "O(n squared) nested loop inefficiency",
    "bug_4_none_comparison": "compare to None Python equality",
    "bug_5_silent_except":   "silently swallowing exceptions Python",
}

# Ground truth — what topic SHOULD appear in top-3
GROUND_TRUTH = {
    "bug_1_mutable_default": "mutable default",
    "bug_2_bare_except":     "bare except",
    "bug_3_nested_loop":     "nested loop",
    "bug_4_none_comparison": "is None",
    "bug_5_silent_except":   "exception handling",
}

In [9]:
def get_api_embeddings(texts, model_id):
    api_url = f"https://router.huggingface.co/hf-inference/models/sentence-transformers/{model_id}/pipeline/feature-extraction"

    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    
    payload = {
        "inputs": texts,
        "options": {"wait_for_model": True}  # ✅ eliminates the 503 retry loop
    }
    
    response = requests.post(api_url, headers=headers, json=payload)
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        return None
    
    result = response.json()
    return np.array(result)  # shape: (N, 384) or (N, 768)

In [ ]:
#Embed query in 768 dim as default embedding of chromadb is 364 dim and vector prepped in embedding-models.ipynb was 768 dim
all_queries = list(BUG_QUERIES.values())
all_embeddings = get_api_embeddings(all_queries, model_ids["MPNet"])


In [ ]:
# --- Reusable audit function ---
def run_audit(collection, query_embeddings, bug_queries, ground_truth, label=""):
    results = []
    print(f"\n{'='*60}")
    print(f"AUDIT: {label}")
    print(f"{'='*60}")
    
    for i, (bug_id, query_text) in enumerate(bug_queries.items()):
        qr = collection.query(
            query_embeddings=[query_embeddings[i].tolist()],
            n_results=3
        )
        gt = list(ground_truth.values())[i]
        hit = False
        hit_ids = []
        
        print(f"\nQuery {i+1}: {query_text}")
        print(f"Ground Truth keyword: '{gt}'")
        for j in range(len(qr['documents'][0])):
            doc = qr['documents'][0][j]
            dist = qr['distances'][0][j]
            rid = qr['metadatas'][0][j]['rule_id']
            type_nm = qr['metadatas'][0][j]['type']
            print(f"[{j}] (distance: {dist}) {type_nm}/{rid}] {doc}")
            match = gt.lower() in doc.lower()
            if match:
                hit = True
                hit_ids.append(rid)
        
        status = f"HIT  (rule_ids: {hit_ids})" if hit else "MISS"
        print(f"  → {status}")
        results.append(hit)
    
    precision = sum(results) / len(results) * 100
    print(f"\nPrecision@3: {sum(results)}/{len(results)} = {precision:.0f}%")
    return results


In [54]:
#Retrieval audit results
retrieval_audit = run_audit(collection, all_embeddings, BUG_QUERIES, GROUND_TRUTH, label="Retrieval audit")


AUDIT: Retrieval audit

Query 1: mutable default argument Python function
Ground Truth keyword: 'mutable default'
[0] (distance: 0.24986830353736877) rule/0] Avoid mutable default arguments in Python functions — use None and set inside the function body
[1] (distance: 0.8455368280410767) rule/4] Mutable default arguments are shared across all calls — this causes hidden state bugs
[2] (distance: 1.448323369026184) rule/5] range(len(x)) is a code smell in Python — enumerate gives you cleaner, more Pythonic code
  → HIT  (rule_ids: [0, 4])

Query 2: bare except clause Python anti-pattern
Ground Truth keyword: 'bare except'
[0] (distance: 1.18316650390625) rule/0] Avoid mutable default arguments in Python functions — use None and set inside the function body
[1] (distance: 1.3448574542999268) rule/5] range(len(x)) is a code smell in Python — enumerate gives you cleaner, more Pythonic code
[2] (distance: 1.5736660957336426) rule/3] Remove unused variables — they add noise and confuse reade

**My Analysis**

The first audit showed distances > 1.0 for bugs 2–5. 

In ChromaDB's L2 space, that means the query vector and the nearest document vector are far apart — the retriever had no genuinely relevant chunk to return. 

It wasn't confused between two similar rules; the rules simply didn't exist. 

The original 8-rule collection only covered mutable defaults, enumerate vs range, context managers, and unused variables — gaps for bare except, None comparison, O(n²), and silent exception swallowing were total.

In [55]:
NEW_RULES = [
    # Bug 2 — bare except
    "Never use a bare except clause — always catch specific exceptions like ValueError or IOError",
    "Bare except catches SystemExit and KeyboardInterrupt too — this masks crashes and makes debugging impossible",

    # Bug 3 — O(n²)
    "Avoid nested loops over the same list — use a set for O(1) lookups to reduce O(n²) to O(n)",
    "Finding duplicates with two nested for loops is O(n²) — convert to a set and check membership instead",

    # Bug 4 — None comparison
    "Always use 'is None' or 'is not None' to check for None — never use == or != with None",
    "Comparing to None with == invokes __eq__ and can return unexpected results — 'is None' checks identity",

    # Bug 5 — silent exception swallowing
    "Never return None silently from an exception handler — either re-raise, log the error, or raise a domain-specific exception",
    "Swallowing exceptions with 'except Exception: return None' hides failures and makes debugging impossible in production",
]

rule_vecs_mpnet = get_api_embeddings(NEW_RULES, model_ids["MPNet"])


In [ ]:
#Logic to extract the next number (assuming format 'rule_N')
total_count = collection.count()

if total_count > 0:
    last_record = collection.get(
        limit=1,
        offset=total_count - 1
    )
    
    last_id = last_record['ids'][0]
    print(f"The last ID is: {last_id}")
    
    try:
        last_num = int(last_id.split('_')[-1])
        next_num = last_num + 1
    except ValueError:
        next_num = total_count # Fallback
else:
    next_num = 0

print(f"Start your next rules from index: {next_num}")



The last ID is: rule_7
Start your next rules from index: 8


In [56]:
#Add new rules in collection
new_ids = [f"rule_{i}" for i in range(next_num, next_num + len(NEW_RULES))]

collection.add(
    embeddings=rule_vecs_mpnet.tolist(),
    documents=NEW_RULES,
    ids=new_ids,
    metadatas=[{"type": "rule", "rule_id": i} for i in range(next_num, next_num + len(NEW_RULES))]
)

In [57]:
#Check if new rules are added 
all_data = collection.get()

for i in range(len(all_data['ids'])):
    doc = all_data['documents'][i]
    metadata = all_data['metadatas'][i]
    rule_id = all_data['ids'][i]
    
    print(f"ID: {rule_id} | Type: {metadata.get('type')} | Content: {doc[:50]}...")

ID: rule_0 | Type: rule | Content: Avoid mutable default arguments in Python function...
ID: rule_1 | Type: rule | Content: Use enumerate() instead of range(len()) when you n...
ID: rule_2 | Type: rule | Content: Always use context managers (with open()) to ensur...
ID: rule_3 | Type: rule | Content: Remove unused variables — they add noise and confu...
ID: rule_4 | Type: rule | Content: Mutable default arguments are shared across all ca...
ID: rule_5 | Type: rule | Content: range(len(x)) is a code smell in Python — enumerat...
ID: rule_6 | Type: rule | Content: File handles not closed with context managers can ...
ID: rule_7 | Type: rule | Content: Dead code including unused bindings should be remo...
ID: rule_8 | Type: rule | Content: Never use a bare except clause — always catch spec...
ID: rule_9 | Type: rule | Content: Bare except catches SystemExit and KeyboardInterru...
ID: rule_10 | Type: rule | Content: Avoid nested loops over the same list — use a set ...
ID: rule_11 | Type: 

In [58]:
# Testing Retrieval audit again
retrieval_audit_afternewrules = run_audit(collection, all_embeddings, BUG_QUERIES, GROUND_TRUTH, label="Retrieval audit after adding new rules")



AUDIT: Retrieval audit after adding new rules

Query 1: mutable default argument Python function
Ground Truth keyword: 'mutable default'
[0] (distance: 0.24986830353736877) rule/0] Avoid mutable default arguments in Python functions — use None and set inside the function body
[1] (distance: 0.8455368280410767) rule/4] Mutable default arguments are shared across all calls — this causes hidden state bugs
[2] (distance: 1.3522605895996094) rule/13] Comparing to None with == invokes __eq__ and can return unexpected results — 'is None' checks identity
  → HIT  (rule_ids: [0, 4])

Query 2: bare except clause Python anti-pattern
Ground Truth keyword: 'bare except'
[0] (distance: 0.7627013325691223) rule/8] Never use a bare except clause — always catch specific exceptions like ValueError or IOError
[1] (distance: 0.9319846630096436) rule/13] Comparing to None with == invokes __eq__ and can return unexpected results — 'is None' checks identity
[2] (distance: 1.0755469799041748) rule/15] Swallo

**My Analysis**

Bugs 1–4 all moved to HIT with distance < 0.5, which is a strong semantic match. 

Bug 5 showed as MISS only because of the ground truth keyword mismatch — the retrieved chunk ("Swallowing exceptions with 'except Exception: return None'...") is the correct rule. Fixing the keyword to "swallowing" resolves this.

In [59]:
GROUND_TRUTH['bug_5_silent_except']='swallowing'

In [60]:
retrieval_audit_ground_truth_correction = run_audit(collection, all_embeddings, BUG_QUERIES, GROUND_TRUTH, label="Retrieval audit after grounf truth correction")



AUDIT: Retrieval audit after grounf truth correction

Query 1: mutable default argument Python function
Ground Truth keyword: 'mutable default'
[0] (distance: 0.24986830353736877) rule/0] Avoid mutable default arguments in Python functions — use None and set inside the function body
[1] (distance: 0.8455368280410767) rule/4] Mutable default arguments are shared across all calls — this causes hidden state bugs
[2] (distance: 1.3522605895996094) rule/13] Comparing to None with == invokes __eq__ and can return unexpected results — 'is None' checks identity
  → HIT  (rule_ids: [0, 4])

Query 2: bare except clause Python anti-pattern
Ground Truth keyword: 'bare except'
[0] (distance: 0.7627013325691223) rule/8] Never use a bare except clause — always catch specific exceptions like ValueError or IOError
[1] (distance: 0.9319846630096436) rule/13] Comparing to None with == invokes __eq__ and can return unexpected results — 'is None' checks identity
[2] (distance: 1.0755469799041748) rule/15]